# Padding deviation of one encoder; for WavLM Base, compared with the original per-layer table

Run at a fixed commit on a Colab runtime with a T4 GPU, after
`notebooks/02_migrate_wavlm_features.ipynb` has passed. The encoder is set in
the cell "What this run does". Features are computed in memory only: the
notebook writes small tables under `tables/<encoder>/` and check tables under
`checks/`, and no feature file.

This notebook measures how far the features of a zero-padded input are from
the features of the unpadded input (`padding.padding_deviation`). Every file
is encoded three ways: padded without an attention mask, padded with one,
and unpadded (the reference). It follows Sections 7 and 7a of
`docs/analysis_plan_hubert.md` (commit `1ecf732`).

One run per encoder, in this order:

1. **`ENCODER = "hubert-base"` (as saved).** The deviation with file-level
   intervals, whether the mask reduces it, and the noise floor. Descriptive
   (Section 7).
2. **`ENCODER = "wavlm-large"`.** The same, and the two predictions of
   Section 7a about the noise floor, per layer and condition, with the
   overall verdict. No classifier is trained and no feature is stored.
3. **`ENCODER = "wavlm-base"`.** The intervals and the noise floor of WavLM
   Base, which the plan asks for and which the run of gate A4 did not
   compute. The tables of gate A4 exist (the gate passed at commit
   `637b4b8`): they are read, not computed again and not written. Gate A4
   itself (below) runs only if its tables do not exist yet.

**The tables of gate A4 are not overwritten.** They are the three tables of
WavLM Base (`padding_deviation.csv`, `padding_deviation_files.csv`,
`padding_deviation_by_amount.csv`) and its five check tables. The per-file
table is the reference against which the samples of the other encoders are
checked. Whenever these tables exist, the run starts by reading the meta
file of each one: it must record the commit `637b4b8`, at which the gate
passed, and the sha256 that the table has now. Otherwise the notebook stops,
before any checkpoint is loaded. Their sha256 is recorded then, and the last
cell stops unless every one is unchanged.

For every encoder, before any checkpoint is loaded, the sha256 of the plan,
of the units file and of its summary are compared with
`docs/preregistration.md`; a difference stops the notebook. The factor 10 of
prediction (a) is read from the code and must be the one the plan states.

The samples of every encoder are the two samples of WavLM Base (500 random
and 500 most padded training files per condition), read from
`tables/wavlm-base/padding_deviation_files.csv`. For HuBERT Base and WavLM
Large the notebook stops unless

- the random sample, drawn again with `numpy.random.default_rng(0)`, is the
  files of WavLM Base in the same order;
- the real length of every sampled file is the `n_real` of WavLM Base: for
  HuBERT Base from its own training shards (and the most padded files, chosen
  again from those shards, are the files of WavLM Base in the same order);
  for WavLM Large, which has no shards, from the audio itself.

Each run starts with a timed check of 25 files that prints an estimate of
the deviation and of the noise floor before the long part.

## Gate A4 (WavLM Base, only while its tables do not exist)

The deviation of WavLM Base is measured on the files of the original check
and compared with the original per-layer table.

The gate is the condition of the analysis plan: on the same files, for both
samples (500 random and 500 most padded training files), both conditions
(untrimmed and trimmed at 40 dB), both mask settings and all 12 layers,
`rel_mean_diff` of the mean-pooled feature differs from the original table by
at most 1e-4. The notebook also requires that the two samples drawn by this
package are the files of the original check, and that every padding number
the manuscript prints is reproduced at the precision it is printed with:
the numbers of the new tables and of the original tables are each rounded
once to the printed digits and must be equal. These are the three entries
of the in-text registry that come from this check (the deviation at layer 4
on the random sample without and with a mask, on untrimmed and trimmed
audio; the median padding of the random sample in the two conditions; the
deviation for less than 1 s and for more than 3 s of padding) and the pair
of the analysis plan for the most padded files (with and without a mask,
trimmed audio, layer 4).

The other columns of the per-layer table and the table by amount of padding
are compared too, with the tolerance of the mapping file, written and shown,
but they do not decide the gate: a largest difference over files and
dimensions, and a ratio built on it, can move by more than 1e-4 between two
runs whose features agree within 1e-4.

The steps:

1. what this run does; the pre-registered files;
2. inputs: the training file list, the samples, the encoder on the GPU and on the CPU;
3. the samples (gate A4: against the files of the original check; otherwise against the samples of WavLM Base, a hard stop);
4. the timed check of 25 files and the estimate;
5. the deviation of the four combinations of condition and sample (about 2,000 files, three encodings each); for WavLM Base with existing tables it is read from them;
6. gate A4 only: the comparison with the original tables, the printed numbers and the summary of the gate (a hard stop);
7. the file-level intervals and whether the mask reduces the deviation;
8. the noise floor (the random sample, unpadded, on the GPU and on the CPU);
9. WavLM Large only: the predictions about the noise floor and their verdict;
10. summary, and the sha256 of the tables of gate A4 (a hard stop).

In a run of gate A4 nothing stops the notebook before the summary of the
gate; every check table of the gate is written first.

In [ ]:
from google.colab import drive, userdata
import os

drive.mount("/content/drive")
assert os.path.ismount("/content/drive"), "Google Drive is not mounted"

### Repository at a fixed commit

Set `COMMIT` to the commit of this repository that is being checked: the
full hash or at least its first 7 characters. The repository is public and
is cloned without a token. If the Colab secret `GITHUB_TOKEN` exists and
this notebook has access to it, the clone uses it (a repository that is not
public needs one); the token is not kept in the clone and is replaced by
`***` in everything git prints.

In [ ]:
import subprocess

REPOSITORY = "Nooralabd/silence-shortcut-wavlm-layers"
COMMIT = ""          # the commit to run: the full hash or at least its first 7 characters
WORKDIR = "/content/silence-shortcut-wavlm-layers"

assert len(COMMIT) >= 7 and set(COMMIT) <= set("0123456789abcdef"), \
    "set COMMIT to the commit that is being checked (full hash or at least its first 7 characters)"
try:
    token = userdata.get("GITHUB_TOKEN") or ""
except Exception:    # the secret does not exist, or this notebook has no access to it
    token = ""       # a public repository is cloned without a token


def git(*arguments):
    """Run git and print what it prints; the token is never printed."""
    result = subprocess.run(["git", *arguments], capture_output=True, text=True)
    output = (result.stdout + result.stderr).strip()
    if token:
        output = output.replace(token, "***")
    if output:
        print(output)
    assert result.returncode == 0, "git failed: see the lines above"
    return output


def clone(repository, folder):
    """Clone a repository, or fetch into an existing clone; a token, if there is one, is not kept in it."""
    address = f"https://{token}@github.com/{repository}.git" if token else f"https://github.com/{repository}.git"
    if not os.path.isdir(folder):
        git("clone", "-q", address, folder)
        git("-C", folder, "remote", "set-url", "origin", f"https://github.com/{repository}.git")
    else:
        git("-C", folder, "fetch", "-q", address, "+refs/heads/*:refs/remotes/origin/*")


clone(REPOSITORY, WORKDIR)
git("-C", WORKDIR, "checkout", "-q", COMMIT)
git("-C", WORKDIR, "rev-parse", "HEAD")
%cd {WORKDIR}

In [ ]:
# Installation. torch, torchaudio and torchcodec are the builds that extracted
# the original features: the pinned versions for CUDA 13.0. If anything pinned
# is missing or has another version, everything is installed in one pass and
# the cell always stops for a restart: a kernel never continues after pip has
# run in it (the runtime has imported numpy before this cell, and a module
# that is already loaded stays loaded when pip replaces it on disk).
# On the run after the restart nothing is installed: the cell checks the
# installed versions and the versions of the modules that the kernel loads.
import importlib
import importlib.metadata
import re
import sys

TORCH_INDEX = "https://download.pytorch.org/whl/cu130"
TORCH_FAMILY = ("torch", "torchaudio", "torchcodec")          # installed from TORCH_INDEX
INSTALLED_ONCE = os.path.join(os.path.dirname(WORKDIR), "installed_once.txt")
OTHERS = os.path.join(os.path.dirname(WORKDIR), "requirements_without_torch.txt")
CONSTRAINT = os.path.join(os.path.dirname(WORKDIR), "torch_constraint.txt")

with open(os.path.join(WORKDIR, "requirements.txt"), encoding="utf-8") as fh:
    requirements = [line.strip() for line in fh if line.strip() and not line.lstrip().startswith("#")]
pins = dict(line.split("==") for line in requirements if "==" in line)
BUILDS = {name: pins[name] + "+" + TORCH_INDEX.rsplit("/", 1)[1] for name in TORCH_FAMILY}


def installed(name):
    try:
        return importlib.metadata.version(name)
    except importlib.metadata.PackageNotFoundError:
        return None


def not_as_pinned():
    """{package: (installed, wanted)} for everything that is missing or has another version."""
    found = {}
    for line in requirements:
        name = re.split(r"[=<>!~ ]", line, maxsplit=1)[0]
        have, want = installed(name), BUILDS.get(name, pins.get(name))
        same = have is not None and (want is None or (have if name in BUILDS else have.split("+")[0]) == want)
        if not same:
            found[name] = (have, want)
    if installed("ssl-silence") is None:
        found["ssl-silence"] = (None, "this repository")
    return found


different = not_as_pinned()
if different:
    assert not os.path.exists(INSTALLED_ONCE), (
        f"the libraries were installed once already and these are still not as pinned: {different}. "
        "Stop: do not restart again")
    imported = sorted(name for name in (*TORCH_FAMILY, "transformers", "ssl_silence") if name in sys.modules)
    assert not imported, (f"{imported} were imported before the installation: "
                          "Runtime > Restart session, then run the notebook again from the top")
    with open(OTHERS, "w", encoding="utf-8") as fh:
        for line in requirements:
            if re.split(r"[=<>!~ ]", line, maxsplit=1)[0] not in TORCH_FAMILY:
                print(line, file=fh)
    with open(CONSTRAINT, "w", encoding="utf-8") as fh:
        for name in TORCH_FAMILY:
            print(f"{name}=={BUILDS[name]}", file=fh)
    # The build is part of the pin (torch==2.11.0+cu130): for a pin without it pip
    # keeps another build of the same version that the runtime ships (+cpu).
    FAMILY_PINS = " ".join(f"{name}=={BUILDS[name]}" for name in TORCH_FAMILY)
    # torchvision of the runtime is built for its own torch and breaks imports
    # with another one; this package does not use it.
    !pip uninstall -q -y torchvision
    !pip install -q {FAMILY_PINS} --index-url {TORCH_INDEX}
    !pip install -q -r {OTHERS} -c {CONSTRAINT}
    !pip install -q --no-deps -e .
    importlib.invalidate_caches()
    # Before the restart: everything on disk is as pinned, so the next run installs nothing.
    different = not_as_pinned()
    assert not different, f"pip did not install these as pinned: {different}: stop"
    with open(INSTALLED_ONCE, "w", encoding="utf-8") as fh:
        print("the pinned libraries were installed in this runtime", file=fh)
    # pip has run in this kernel: it always stops here.
    raise RuntimeError("The pinned libraries were installed. "
                       "Runtime > Restart session, then run the notebook again from the top")

# Nothing was installed in this run. The modules that this kernel loads must
# be the pinned versions too, not only the files on disk.
LOADED = {"numpy": "numpy", "scipy": "scipy", "pandas": "pandas", "scikit-learn": "sklearn",
          "torch": "torch", "torchaudio": "torchaudio", "transformers": "transformers"}
for name, module in LOADED.items():
    loaded = importlib.import_module(module).__version__
    assert loaded.split("+")[0] == pins[name], (
        f"{module} {loaded} is loaded in this kernel and the pin is {pins[name]}: it was imported before pip "
        "replaced it. Runtime > Restart session, then run the notebook again from the top")

import torch
import torchaudio
import transformers

print("loaded:", {name: sys.modules[module].__version__ for name, module in LOADED.items()})
print("torch", torch.__version__, "| CUDA of the build:", torch.version.cuda)
print("torchaudio", torchaudio.__version__, "| torchcodec", installed("torchcodec"),
      "| transformers", transformers.__version__)
if os.path.join(WORKDIR, "src") not in sys.path:
    sys.path.insert(0, os.path.join(WORKDIR, "src"))   # a running kernel does not see a new editable install

In [ ]:
import subprocess
import sys

import pandas as pd

from ssl_silence import config as cfg
from ssl_silence import provenance, verify

pd.set_option("display.max_rows", 200, "display.width", 200)
HEAD = provenance.git_commit()     # the full hash, with "-dirty" if a tracked file was changed
assert HEAD.startswith(COMMIT) and not HEAD.endswith("-dirty"), (HEAD, COMMIT)
print("commit:", HEAD)
print("root:", cfg.ROOT)

In [ ]:
# The quota of the Google account is read from the Drive API (this asks for
# access to the account); the mounted folder does not report it.
QUOTA = cfg.drive_quota()
print("Google Drive: {usage_gb:.1f} GB used of {limit_gb:.1f} GB, {free_gb:.1f} GB free".format(**QUOTA),
      "(GB = 10^9 bytes; required: %d GB free)" % cfg.MIN_FREE_GB)
cfg.require_free_space(quota=QUOTA)
print("local disk of the runtime: %.1f GB free" % cfg.local_free_space_gb())

### Mapping file

`legacy_map.yaml` names the original folders, files and labels. It is kept on
Drive only, at `<root>/migration/legacy_map.yaml`, and must have no empty
value (`verify.load_map` stops otherwise).

In [ ]:
MAP_PATH = os.path.join(cfg.migration_dir(), "legacy_map.yaml")
assert os.path.exists(MAP_PATH), f"upload legacy_map.yaml to {MAP_PATH}"
legacy = verify.load_map(MAP_PATH)
FORBIDDEN = legacy["forbidden"]
CHECKS = cfg.checks_dir()
checks = {}


def record(name, table, stop=True):
    """Write a check table to Drive, show it, and stop if a row fails."""
    path = verify.write_check(table, name, CHECKS, FORBIDDEN)
    checks[name] = table
    display(table)
    if stop and "ok" in table.columns:
        verify.require_ok(table, name)
    return path

### What this run does

`ENCODER` is the only value to set.

In [ ]:
from ssl_silence import encoders, padding, replication

ENCODER = "hubert-base"    # one run per encoder, in this order: "hubert-base", "wavlm-large", "wavlm-base"

REFERENCE = padding.REFERENCE_ENCODER                       # its two samples are the samples of every encoder
PREDICTIONS = ENCODER == padding.PREDICTED_ENCODER          # the predictions of Section 7a of the plan
assert ENCODER in cfg.encoders(), f"ENCODER is {ENCODER!r}: set it to one of {list(cfg.encoders())}"
HAS_SHARDS = ENCODER in cfg.experiments()["experiments"]    # features of this encoder were extracted

# The tables of gate A4: never computed again and never written while they exist.
GATE_FILES = [os.path.join(cfg.tables_dir(REFERENCE), f"{name}.csv") for name in padding.GATE_TABLES] \
    + [os.path.join(CHECKS, f"{name}.csv") for name in padding.GATE_CHECKS]
existing = [path for path in GATE_FILES if os.path.exists(path)]
assert len(existing) in (0, len(GATE_FILES)), \
    f"only {len(existing)} of the {len(GATE_FILES)} tables of gate A4 exist: {existing}"
GATE = ENCODER == REFERENCE and not existing                # gate A4 runs only while its tables do not exist
REUSE = ENCODER == REFERENCE and bool(existing)             # WavLM Base with its tables: they are read
assert GATE or existing, f"the tables of gate A4 do not exist: run this notebook for {REFERENCE} first"
GATE_SHA256 = {path: provenance.sha256_file(path) for path in existing}
if existing:
    # They must be the tables that gate A4 wrote: the commit and the sha256 of each meta file (a hard stop).
    gate_files = padding.gate_files_check(GATE_SHA256)
    for row in gate_files.itertuples():
        print(os.path.basename(row.file), "| commit of its meta file:", row.git_commit)
    record(f"padding_gate_commit_{ENCODER}", gate_files)
print("this run:", ENCODER, "|",
      "gate A4 (comparison with the original tables), then the intervals and the noise floor" if GATE
      else "the intervals and the noise floor; the deviation is read from the tables of gate A4" if REUSE
      else "the deviation with intervals and the noise floor" + (", and the predictions about the floor" if PREDICTIONS else ""))
print("tables of gate A4:", "none yet" if GATE else f"{len(GATE_SHA256)} exist, written at commit {padding.GATE_COMMIT}; their sha256 is recorded and must not change")

### The pre-registered files

Before any checkpoint is loaded: the sha256 of `docs/analysis_plan_hubert.md`,
`docs/replication_units.csv` and `docs/replication_units_summary.csv` of the
checked-out commit against `docs/preregistration.md` (commit `1ecf732`). A
difference stops the notebook. The factor of prediction (a), the size and
the seed of the samples and the generator stream of the intervals are those
of the code; the factor must be the one the plan states.

The checkpoints must be downloaded to the local disk of the runtime, not to
Drive.

In [ ]:
record(f"preregistration_padding_{ENCODER}", replication.preregistration_check(WORKDIR))
with open(os.path.join(WORKDIR, replication.PREREGISTERED_PLAN), encoding="utf-8") as fh:
    PLAN = fh.read()
assert "upper <= 10.0 * floor_upper" in PLAN and padding.FLOOR_FACTOR == 10.0, \
    "the factor of prediction (a) is not the factor of the pre-registered plan"
assert "numpy.random.default_rng(0)" in PLAN and (padding.SAMPLE_SEED, padding.SAMPLE_SIZE) == (0, 500)
assert padding.INTERVAL_SEED == (cfg.BOOTSTRAP_SEED, 101) and cfg.N_BOOT == 1_000
HF_CACHE = encoders.cache_dir()
assert not cfg.on_drive(HF_CACHE), f"the Hugging Face cache is on Drive: {HF_CACHE}"
print("the pre-registered files are unchanged | factor of prediction (a):", padding.FLOOR_FACTOR,
      "| intervals:", cfg.N_BOOT, "resamples, generator stream", padding.INTERVAL_SEED)
print("Hugging Face cache (local disk of the runtime):", HF_CACHE)

### Inputs

The ASVspoof 2019 LA archive is unpacked to the local disk for the training
audio. In a run of gate A4 the files of the original check are read from the
migrated copy of its per-file table
(`migration/original/files/padding_per_file.csv`); in every other run the
samples are read from the per-file table of WavLM Base
(`tables/wavlm-base/padding_deviation_files.csv`). The checkpoint is
downloaded from Hugging Face at the revision of the registry, compared with
its entry, and loaded twice: on the GPU, and on the CPU for the noise floor.

In [ ]:
import torch

from ssl_silence import datasets, shards

assert torch.cuda.is_available(), "no GPU: Runtime > Change runtime type > T4 GPU, then run again from the top"
DEVICE = "cuda"
CONDITIONS = ("clean", "trim40")
ORIGINAL = os.path.join(cfg.migration_dir(), "original")
OUT = cfg.tables_dir(ENCODER)

datasets.ensure_asvspoof(cfg.la_zip())
items = datasets.items("train")
print("train", datasets.check_items(items, "train"))
if GATE:
    original_files = pd.read_csv(os.path.join(ORIGINAL, "files", "padding_per_file.csv"), dtype={"audio_id": str})
    print("rows of the original per-file table:", len(original_files))
else:
    REFERENCE_FILES = os.path.join(cfg.tables_dir(REFERENCE), "padding_deviation_files.csv")
    assert REFERENCE_FILES in GATE_SHA256, REFERENCE_FILES
    reference_files = pd.read_csv(REFERENCE_FILES, dtype={"audio_id": str})
    print("rows of the per-file table of", REFERENCE + ":", len(reference_files))

entry = cfg.encoder(ENCODER)
model = encoders.load_encoder(ENCODER, DEVICE)
cpu_model = encoders.load_encoder(ENCODER, "cpu")      # the same weights on the CPU, for the noise floor
described = {key: str(value) for key, value in encoders.describe(model).items()}
assert not PREDICTIONS or described["feat_extract_norm"] == "layer", described
record("padding_encoder" if GATE else f"padding_encoder_{ENCODER}",
       pd.DataFrame([{"encoder": ENCODER, "hf_id": entry["hf_id"], "revision": entry["revision"],
                      "gpu": provenance.gpu_name(DEVICE), **described}]), stop=False)

### Samples

**Gate A4.** The random sample is drawn with the fixed seed of
`padding.random_items`; the most padded files are chosen from the input
lengths stored in the training shards of each condition. Both must be the
files of the original check. The deviation below is measured on the files of
the original check, in its order.

**Every other run** (HuBERT Base, WavLM Large, and WavLM Base with its
tables). The samples are those of WavLM Base, in its order. A hard stop unless the random sample drawn again is the same files in
the same order, and the real length of every sampled file equals the
`n_real` of WavLM Base. HuBERT Base and WavLM Base: the lengths of the encoder's own training shards,
and the most padded files chosen again from them. WavLM Large: the lengths
computed from the audio (it has no shards, so its most padded files are not
chosen again; their lengths are compared).

In [ ]:
samples, rows = {}, []
for condition in CONDITIONS:
    if GATE:
        lengths = shards.shard_lengths(cfg.feature_dir(ENCODER, "unpadded", condition), splits=("train",))
        drawn = {"random": padding.random_items(items),
                 "most_padded": padding.most_padded_items(items, padding.padding_from_lengths(lengths))}
        for sample in padding.SAMPLES:
            ids = padding.sample_ids(original_files, legacy["conditions"][condition], sample)
            samples[(condition, sample)] = padding.items_with_ids(items, ids)
            rows.append({"condition": condition, "sample": sample, **padding.compare_samples(drawn[sample], ids)})
        continue
    # Another encoder: the samples of the reference encoder, checked against what this encoder has.
    chosen = {sample: padding.items_with_ids(items, padding.sample_ids(reference_files, condition, sample))
              for sample in padding.SAMPLES}
    redrawn = {"random": padding.random_items(items), "most_padded": None}     # numpy.random.default_rng(0)
    if HAS_SHARDS:
        lengths = shards.shard_lengths(cfg.feature_dir(ENCODER, "unpadded", condition), splits=("train",))
        known, lengths_from = padding.stored_lengths(lengths), f"training shards of {ENCODER}"
        redrawn["most_padded"] = padding.most_padded_items(items, padding.padding_from_lengths(lengths))
    else:
        both = pd.concat(list(chosen.values()), ignore_index=True).drop_duplicates("audio_id")
        known, lengths_from = padding.lengths_from_audio(both, condition), "audio"
    for sample in padding.SAMPLES:
        samples[(condition, sample)] = chosen[sample]
        rows.append(padding.check_reference_sample(reference_files, condition, sample, known, lengths_from,
                                                   redrawn[sample]))
if GATE:
    record("padding_samples", pd.DataFrame(rows), stop=False)
else:
    record(f"padding_samples_{ENCODER}", pd.DataFrame(rows))      # a hard stop

### Timed check

The first 25 files of the random sample of the untrimmed condition are
measured once for the deviation and once for the noise floor. Their tables
are discarded. For WavLM Base with its tables only the noise floor is
timed, because the deviation is not measured. The estimate is printed before the long part; the first files
include the start-up of the GPU, so it is on the high side.

In [ ]:
import time

timed = samples[(CONDITIONS[0], "random")].head(padding.TIMING_FILES)
rows = []
if not REUSE:
    start = time.perf_counter()
    padding.padding_deviation(model, timed, DEVICE, CONDITIONS[0], padding.TIMING)
    rows.append({"step": "deviation", "files_timed": len(timed), "seconds": time.perf_counter() - start,
                 "files": sum(len(chosen) for chosen in samples.values())})
start = time.perf_counter()
padding.noise_floor(model, DEVICE, cpu_model, "cpu", timed, CONDITIONS[0], padding.TIMING)
rows.append({"step": "noise floor", "files_timed": len(timed), "seconds": time.perf_counter() - start,
             "files": sum(len(samples[(condition, "random")]) for condition in CONDITIONS)})
timing = pd.DataFrame(rows)
timing["estimated_minutes"] = timing["seconds"] / timing["files_timed"] * timing["files"] / 60
record(f"padding_timing_{ENCODER}", timing, stop=False)
print("estimate for", ENCODER + ":", round(float(timing["estimated_minutes"].sum()), 1), "minutes |",
      dict(zip(timing["step"], timing["estimated_minutes"].round(1))))

### Deviation

For WavLM Base with its tables the deviation is not measured: the per-layer
and the per-file table of gate A4 are read, and nothing is written here.

In [ ]:
if REUSE:
    per_layer = pd.read_csv(os.path.join(OUT, "padding_deviation.csv"))
    per_file = pd.read_csv(os.path.join(OUT, "padding_deviation_files.csv"), dtype={"audio_id": str})
    print("read from the tables of gate A4, not measured | per-layer rows:", len(per_layer),
          "| files:", len(per_file))
else:
    layer_tables, file_tables = [], []
    for (condition, sample), chosen in samples.items():
        layer_table, file_table = padding.padding_deviation(model, chosen, DEVICE, condition, sample)
        layer_tables.append(layer_table)
        file_tables.append(file_table)
        print(condition, sample, "| files:", len(file_table), "| skipped:", int(layer_table["n_skipped"].iloc[0]),
              "| median padding (s):", round(float(file_table["pad_s"].median()), 2))
    per_layer = pd.concat(layer_tables, ignore_index=True)
    per_file = pd.concat(file_tables, ignore_index=True)
    by_amount = padding.deviation_by_padding(per_file, per_layer)
    for name, table in (("padding_deviation", per_layer), ("padding_deviation_files", per_file),
                        ("padding_deviation_by_amount", by_amount)):
        provenance.write_table(table, os.path.join(OUT, f"{name}.csv"), encoder=ENCODER, revision=entry["revision"])
    published = padding.published_rows(per_layer)
    display(published[published["layer"] == 4].pivot_table(index=["sample", "condition"], columns="comparison",
                                                           values="rel_mean_diff"))

### Comparison with the original tables (gate A4 only)

The rows of the two mask settings against the reference, mean-pooled
feature. One row per compared column, with its largest absolute difference
over all rows of the table.

In [ ]:
if GATE:
    deviation = verify.compare_with_legacy(published, "padding_deviation", legacy, ORIGINAL)
    record("table_padding_deviation", deviation, stop=False)
    amount = verify.compare_with_legacy(padding.published_rows(by_amount), "padding_deviation_by_amount", legacy, ORIGINAL)
    record("table_padding_deviation_by_amount", amount, stop=False)
else:
    print("not a run of gate A4: nothing is compared with the original tables")

### Printed numbers (gate A4 only)

`value` is the number from the tables of this run, `reference_value` the
number from the original tables (the per-layer table, the table by amount of
padding and the per-file table, in the layout of this package); `printed`
and `reference_printed` are both rounded once to `digits` decimals.

In [ ]:
if GATE:
    from ssl_silence.paper import intext

    back = {original: condition for condition, original in legacy["conditions"].items()}
    new_tables = intext.Tables.of({intext.PADDING_DEVIATION: per_layer, intext.PADDING_BY_AMOUNT: by_amount,
                                   intext.PADDING_PER_FILE: per_file})
    original_tables = intext.Tables.of({
        intext.PADDING_DEVIATION: verify.read_legacy("padding_deviation", legacy, ORIGINAL),
        intext.PADDING_BY_AMOUNT: verify.read_legacy("padding_deviation_by_amount", legacy, ORIGINAL),
        intext.PADDING_PER_FILE: original_files.assign(condition=original_files["condition"].map(back))})
    printed = intext.compare_printed(new_tables, original_tables, intext.padding_entries())
    record("padding_printed_numbers", printed, stop=False)
else:
    print("not a run of gate A4: no printed numbers")

### Summary of gate A4 (only in a run of the gate)

In [ ]:
if GATE:
    compared = {name: checks[name] for name in ("padding_samples", "table_padding_deviation",
                                                "table_padding_deviation_by_amount", "padding_printed_numbers")}
    display(verify.summarise_checks(compared))
    print("tables written to", OUT, "| check tables written to", CHECKS)
    gate = deviation[deviation["column"] == "rel_mean_diff"]
    assert len(gate) == 1 and int(gate["n_matched"].iloc[0]) == 2 * 2 * 2 * entry["n_layers"], gate
    verify.require_ok(checks["padding_samples"], "padding_samples")
    verify.require_ok(gate, "rel_mean_diff of padding_deviation")
    assert len(printed) == 10, printed
    verify.require_ok(printed, "padding numbers at the printed precision")
    others = bool(deviation["ok"].all() and amount["ok"].all())
    print("gate A4 passed:", ENCODER, "| rows compared:", int(gate["n_matched"].iloc[0]),
          "| largest difference of rel_mean_diff:", float(gate["max_abs_diff"].iloc[0]),
          "| printed numbers reproduced:", len(printed),
          "| every other compared column within its tolerance:", others)
else:
    print("not a run of gate A4: the gate is not evaluated")

### Intervals, and whether the mask reduces the deviation

File-level percentile intervals from 1,000 resamples of the files of each
condition and sample; the numerator and the denominator are recomputed in
each resample, and the same resamples give the paired difference with mask
minus without mask (`padding_deviation_intervals.csv`).

"The mask reduces the deviation" is the definition of the plan: the paired
difference is negative and its interval excludes zero, on the random sample
(`padding_mask_reduces.csv`). The most padded sample is in the intervals
table and is reported as observed.

In [ ]:
intervals = padding.deviation_intervals(per_file)
reduces = padding.mask_reduces_deviation(intervals)
for name, table in (("padding_deviation_intervals", intervals), ("padding_mask_reduces", reduces)):
    provenance.write_table(table, os.path.join(OUT, f"{name}.csv"), encoder=ENCODER, revision=entry["revision"])
display(reduces)

### Noise floor

The random sample of each condition is encoded unpadded twice, on the GPU and
on the CPU (float32). The floor is the deviation of the CPU features from the
GPU features, with the definition and the file-level interval of the
deviation (`padding_noise_floor.csv`, `padding_noise_floor_files.csv`,
`padding_noise_floor_intervals.csv`).

In [ ]:
layer_tables, file_tables = [], []
for condition in CONDITIONS:
    layer_table, file_table = padding.noise_floor(model, DEVICE, cpu_model, "cpu",
                                                  samples[(condition, "random")], condition, "random")
    layer_tables.append(layer_table)
    file_tables.append(file_table)
    print(condition, "| files:", len(file_table), "| skipped:", int(layer_table["n_skipped"].iloc[0]))
floor_layer = pd.concat(layer_tables, ignore_index=True)
floor_file = pd.concat(file_tables, ignore_index=True)
floor_intervals = padding.deviation_intervals(floor_file, comparisons=("floor",), difference=None)
for name, table in (("padding_noise_floor", floor_layer), ("padding_noise_floor_files", floor_file),
                    ("padding_noise_floor_intervals", floor_intervals)):
    provenance.write_table(table, os.path.join(OUT, f"{name}.csv"), encoder=ENCODER, revision=entry["revision"])
display(floor_intervals[floor_intervals["statistic"] == "mean"])

### Predictions about the noise floor (WavLM Large only)

Section 7a of the plan, on the random sample, mean-pooled feature, per layer
and condition (`padding_floor_predictions.csv`):

- (a) with an attention mask the upper bound of the interval of the
  deviation is at most 10 times the upper bound of the interval of the
  floor, computed in float64 as `upper <= 10.0 * floor_upper`; with a floor
  upper bound of 0 the deviation must be 0;
- (b) without an attention mask the lower bound of the interval of the
  deviation is above the upper bound of the interval of the floor.

Each prediction holds if it is met at all 24 layers in both conditions. The
verdict is written as a check table and reported either way: it does not
stop the notebook. The most padded sample is not part of the predictions.

In [ ]:
if PREDICTIONS:
    predictions = padding.floor_predictions(intervals, floor_intervals)
    verdict = padding.prediction_verdict(predictions, entry["n_layers"], CONDITIONS)
    provenance.write_table(predictions, os.path.join(OUT, "padding_floor_predictions.csv"),
                           encoder=ENCODER, revision=entry["revision"])
    record(f"padding_predictions_{ENCODER}", verdict, stop=False)
    display(predictions)
    for row in verdict.itertuples():
        print(f"prediction ({row.prediction})", "holds" if row.holds else "does not hold",
              f"| met in {row.n_met} of {row.n_cells} layers x conditions")
else:
    print("no prediction is stated for", ENCODER)

### Summary

The last check: the sha256 of every table of gate A4 that existed at the
start of the run is computed again and must be unchanged (a hard stop).

In [ ]:
layers = set(per_layer["layer"])
assert set(intervals["layer"]) == layers and set(floor_intervals["layer"]) == layers
assert set(zip(per_file["condition"], per_file["sample"])) == set(samples)
assert set(floor_file["condition"]) == set(CONDITIONS)
verify.require_ok(checks["padding_samples" if GATE else f"padding_samples_{ENCODER}"], "samples")
if not GATE:
    record(f"padding_gate_tables_{ENCODER}", pd.DataFrame([
        {"file": path, "sha256_at_start": before, "sha256_at_end": provenance.sha256_file(path)}
        for path, before in GATE_SHA256.items()]).assign(
            ok=lambda table: table["sha256_at_start"] == table["sha256_at_end"]))      # a hard stop
print("tables written to", OUT, "| check tables written to", CHECKS)
print("padding run finished:", ENCODER, "| files measured:", "none (read)" if REUSE else len(per_file),
      "| files of the noise floor:", len(floor_file),
      "| tables of gate A4:", "written by this run" if GATE else f"{len(GATE_SHA256)} unchanged",
      "| predictions:", {row.prediction: bool(row.holds) for row in verdict.itertuples()} if PREDICTIONS
      else "none stated for this encoder")